# 2. Auditar y congelar el conjunto real
Ejecuta la primera parte para obtener el inventario. Completa la planilla con la trazabilidad de las capturas y vuelve a ejecutar. La clase normal exige confirmación explícita del anotador.

In [ ]:
from pathlib import Path
from IPython import get_ipython

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)


def load_notebook(name):
    path = ROOT / "notebooks" / "shared" / (name + ".ipynb")
    get_ipython().run_line_magic("run", f'-i "{path.as_posix()}"')


load_notebook("workspace")

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from pycocotools.coco import COCO
from tempfile import TemporaryDirectory

base = dataset_location()
raw = base / "raw"
catalog = {}
inventory = []
for split in ("train", "valid", "test"):
    coco = COCO(str(raw / split / "_annotations.coco.json"))
    catalog[split] = coco
    print(split, {key: cat["name"] for key, cat in coco.cats.items()})
    for item in coco.imgs.values():
        relative = Path(split) / item["file_name"]
        image = (raw / relative).resolve()
        if not image.is_relative_to((raw / split).resolve()) or not image.is_file():
            raise ValueError(f"Ruta de imagen inválida: {relative}")
        inventory.append(
            {"file": relative.as_posix(), "split": split, "image_id": item["id"]}
        )
frame = pd.DataFrame(inventory).sort_values("file")
if frame.file.duplicated().any():
    raise ValueError("Hay nombres de imagen duplicados.")
review = base / "review.csv"
if not review.exists():
    frame[["file"]].assign(group_id="", origin="", label="").to_csv(
        review, index=False, encoding="utf-8-sig"
    )
print("Completa la planilla:", review)

## Máscaras de defectos
Copia los nombres exactos impresos arriba. Incluye cada categoría anotada en una de las dos listas. Una categoría que delimita toda la tapa se puede excluir explícitamente.

In [ ]:
defect_classes = []
ignored_classes = []

In [ ]:
audit = pd.read_csv(review, dtype=str, keep_default_na=False, encoding="utf-8-sig")
rows = frame.merge(audit, on="file", how="outer", indicator=True, validate="one_to_one")
if not rows["_merge"].eq("both").all():
    raise ValueError("Las filas de la planilla deben coincidir con la exportación.")
if not rows.origin.eq("real").all() or not rows.label.isin(["normal", "defect"]).all():
    raise ValueError(
        "Confirma el origen real y la etiqueta normal o defect de cada imagen."
    )
rows["group_id"] = rows.group_id.str.strip()
if rows.group_id.eq("").any() or rows.groupby("group_id").split.nunique().max() != 1:
    raise ValueError(
        "Cada pieza, lote o sesión relacionada necesita un grupo exclusivo de una partición."
    )
if not defect_classes or set(defect_classes) & set(ignored_classes):
    raise ValueError("Revisa la selección de clases de defectos.")
for split in ("valid", "test"):
    if set(rows.loc[rows.split == split, "label"]) != {"defect", "normal"}:
        raise ValueError(f"{split} necesita imágenes normales y defectuosas.")
if not ((rows.split == "train") & (rows.label == "normal")).any():
    raise ValueError("TRAIN requiere tapas normales.")
frozen = base / "frozen"
with TemporaryDirectory(dir=base) as staging:
    target = Path(staging) / "frozen"
    (target / "masks").mkdir(parents=True)
    records = []
    for row in rows.sort_values("file").itertuples():
        coco = catalog[row.split]
        item = coco.imgs[row.image_id]
        path = raw / row.file
        with Image.open(path) as image:
            if image.size != (item["width"], item["height"]):
                raise ValueError(f"Dimensiones incorrectas en {row.file}")
            pixel_hash = hashlib.sha256(image.convert("RGB").tobytes()).hexdigest()
        mask = np.zeros((item["height"], item["width"]), dtype=np.uint8)
        for ann in coco.imgToAnns[row.image_id]:
            name = coco.cats[ann["category_id"]]["name"]
            if name in ignored_classes:
                continue
            if name not in defect_classes or not ann.get("segmentation"):
                raise ValueError(
                    f"Clase desconocida o máscara ausente en {row.file}: {name}"
                )
            mask |= coco.annToMask(ann)
        if bool(mask.any()) != (row.label == "defect"):
            raise ValueError(
                f"La etiqueta revisada contradice la máscara de {row.file}"
            )
        mask_name = f"masks/{row.split}-{row.image_id}.png"
        Image.fromarray(mask * 255).save(target / mask_name)
        records.append(
            dict(
                file=row.file,
                split=row.split,
                group_id=row.group_id,
                label=row.label,
                mask=mask_name,
                image_hash=digest(path),
                pixel_hash=pixel_hash,
                mask_hash=digest(target / mask_name),
            )
        )
    manifest = pd.DataFrame(records)
    if manifest.groupby("pixel_hash").split.nunique().max() > 1:
        raise ValueError("Hay imágenes idénticas en particiones diferentes.")
    manifest.to_csv(target / "manifest.csv", index=False)
    save_json(
        target / "source.json",
        {
            "version": base.relative_to(DATA).as_posix(),
            "format": "coco-segmentation",
            "classes": sorted(defect_classes),
            "ignored": sorted(ignored_classes),
            "annotations": {
                split: digest(raw / split / "_annotations.coco.json")
                for split in catalog
            },
            "manifest": digest(target / "manifest.csv"),
        },
    )
    if frozen.exists():
        if (frozen / "source.json").read_bytes() != (
            target / "source.json"
        ).read_bytes():
            raise ValueError(
                "Esta versión ya está congelada con otro contenido. Genera una versión nueva."
            )
    else:
        target.rename(frozen)
print(manifest.groupby(["split", "label"]).size(), digest(frozen / "manifest.csv"))